In [ ]:
# Plot-spatial-clone-and-sep-celltype-in-cell-level
# Imports
from pathlib import Path
import gc
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from skimage.transform import resize

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(x, **kwargs):
        return x

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["font.family"] = "Arial"
mpl.rcParams["text.usetex"] = False

In [ ]:
 
project_dir = Path("/Volumes/B/Projects/2025Project/SpatialAnalysis/Project_ICC")

base_dir = project_dir / "Data/ST/Samples"
fig_dir = project_dir / "Figure/ST/InferCNV/Clone_SEP_Spatial"
legend_dir = fig_dir / "Legends"

mp_table_path = project_dir / "Data/ST/SEP/All_hex_MP_scores_for_h5ad.csv"

fig_dir.mkdir(parents=True, exist_ok=True)
legend_dir.mkdir(parents=True, exist_ok=True)

celltype_col = "CellType"
sep_col = "SEP"
clone_col = "Clone"
match_col = "matched_hex15_id"

spatial_key = "spatial"
bg_scale = 0.1
bg_alpha = 0.2
spot_size = 0.3
dpi = 500

celltype_colors = {
    "Bcell": "#ba55d3",
    "Endothelial": "#1E90FF",
    "Epithelial": "#808080",
    "Fibroblast": "#BD8F8F",
    "Myeloid": "#FFD500",
    "Tcell": "#FF0000",
    "Hepatocyte": "#1e8157",
    "Hepatocytes": "#1e8157",
}
 
clone_colors = {
    "A": "#f3b0b3",
    "B": "#5e8fac",
    "C": "#FEDD6F",
    "D": "#F89155",
    "E": "#41A263",
    "F": "#82539e",
    "G": "#ac7447", 
}

SEP_colors = {
    "M_SEP1": "#e41a1c", "M_SEP2": "#2373F4", "M_SEP3": "#FCAD38",
    "M_SEP4": "#b3de69", "M_SEP5": "#A290B7", 
}

In [ ]:
 
def make_lowres_rgba_he_background(
    adata,
    image_key="staining_image",
    mask_key="final_mask",
    bg_alpha=0.2,
    bg_scale=0.1,
):
 
    img = np.asarray(adata.uns[image_key])
    if img.ndim == 3 and img.shape[-1] == 4:
        img = img[..., :3]
    img = img.transpose((1, 0, 2)) if img.ndim == 3 else img.T
    img = img.astype(np.float32)
    if np.nanmax(img) > 1:
        img = img / 255.0

    mask = np.asarray(adata.uns[mask_key]).astype(bool)
    if mask.shape != img.shape[:2]:
        if mask.T.shape == img.shape[:2]:
            mask = mask.T

    alpha = np.zeros(img.shape[:2], dtype=np.float32)
    alpha[mask] = bg_alpha
    img_rgba = np.concatenate([img, alpha[..., None]], axis=-1)

    H, W = img_rgba.shape[:2]
    low_H = max(1, int(H * bg_scale))
    low_W = max(1, int(W * bg_scale))

    img_low = resize(
        img_rgba,
        (low_H, low_W, 4),
        preserve_range=True,
        anti_aliasing=True,
    ).astype(np.float32)

    return img_low, low_W, low_H, bg_scale


def add_lowres_spatial(
    adata,
    scale,
    spatial_key="spatial",
    lowres_key="spatial_lowres",
):
    if spatial_key not in adata.obsm:
        raise KeyError(f"adata.obsm['{spatial_key}'] not found")

    adata.obsm[lowres_key] = (
        np.asarray(adata.obsm[spatial_key]).copy() * scale
    )
    return adata


def is_unassigned(series):
    return series.astype(str).str.strip().str.lower().eq("unassigned")


def filter_for_plot(adata, column):
    if column not in adata.obs.columns:
        raise KeyError(f"'{column}' not found in adata.obs")

    x = adata.obs[column]
    keep = x.notna() & (~is_unassigned(x))
    return adata[keep].copy()

 

def plot_spatial_lowres_rgba(
    adata,
    color_col,
    color_dict,
    img_low,
    W_low,
    H_low,
    out_prefix,
    outdir,
    basis="spatial_lowres",
    spot_size=0.3,
    dpi=500,
):
    if adata.n_obs == 0:
        print(f"[Skip plot] {out_prefix}: no spots after filtering")
        return None
 
    coords = np.asarray(adata.obsm[basis])
    labels = adata.obs[color_col].astype(str)
 
    point_colors = labels.map(
        lambda x: color_dict.get(x, "#D9D9D9")
    ).values

    fig = plt.figure(
        figsize=(W_low / dpi, H_low / dpi),
        dpi=dpi,
        frameon=False,
    )
    ax = fig.add_axes([0, 0, 1, 1])

    ax.imshow(
        img_low,
        extent=[0, W_low, H_low, 0],
        interpolation="bilinear",
    )

    ax.scatter(
        coords[:, 0],
        coords[:, 1],
        c=point_colors,
        s=spot_size,
        linewidths=0,
        rasterized=True,
    )

    ax.set_xlim(0, W_low)
    ax.set_ylim(H_low, 0)
    ax.set_aspect("equal")
    ax.axis("off")

    out_pdf = Path(outdir) / f"{out_prefix}.pdf"
    fig.savefig(out_pdf, dpi=dpi, pad_inches=0, facecolor="white", edgecolor="none",)
    plt.close(fig)

    return out_pdf


def save_discrete_legend(
    color_dict,
    out_prefix,
    out_dir,
    title,
    ncol=1,
    dpi=300,
):
    labels = list(color_dict.keys())
    nrow = int(np.ceil(len(labels) / ncol))
    fig_height = max(1.5, nrow * 0.33)

    fig, ax = plt.subplots(
        figsize=(4.2 if ncol > 1 else 2.8, fig_height),
        dpi=dpi,
    )
    ax.axis("off")

    handles = [
        Line2D(
            [0], [0],
            marker="o",
            linestyle="",
            markerfacecolor=color_dict[x],
            markeredgecolor="none",
            markersize=8,
            label=x,
        )
        for x in labels
    ]

    legend = ax.legend(
        handles=handles,
        labels=labels,
        title=title,
        loc="center",
        frameon=False,
        ncol=ncol,
        fontsize=9 if ncol > 1 else 10,
        handletextpad=0.5,
        columnspacing=1.2,
    )

    legend.get_title().set_fontsize(11)
    legend.get_title().set_fontweight("bold")

    out_pdf = Path(out_dir) / f"{out_prefix}.pdf"
    fig.savefig(
        out_pdf,
        bbox_inches="tight",
        dpi=dpi,
        transparent=True,
        pad_inches=0.05,
    )
    plt.close(fig)

    return out_pdf

In [ ]:
# Load SEP mapping table
mp_table = pd.read_csv(mp_table_path)
mp_use = (mp_table[["SpotID", "SEP"]].dropna(subset=["SpotID"]).copy())
mp_use["_match_id"] = (mp_use["SpotID"].astype(str).str.strip())

sep_map = (mp_use.drop_duplicates("_match_id").set_index("_match_id")["SEP"])
print(f"SEP mapping entries: {len(sep_map):,}")

In [ ]:
 
h5ad_paths = sorted(base_dir.glob("*/Integrated/cell_adata.h5ad"))
sample_records = [{"Sample": p.parent.parent.name,
                   "h5ad_path": p,}
    for p in h5ad_paths
]
print(f"Found {len(sample_records)} samples:")
print([x["Sample"] for x in sample_records])

In [ ]:
 
annotation_tables = []
run_summary = []

for rec in tqdm(sample_records, desc="Samples"):

    sample_id = rec["Sample"]
    h5ad_path = rec["h5ad_path"]
    outdir = fig_dir / sample_id
    outdir.mkdir(parents=True, exist_ok=True)

    print("\n" + "=" * 80)
    print(f"[Sample] {sample_id}")
    print("=" * 80)

    try:
        adata = sc.read_h5ad(h5ad_path)

        required_cols = [
            celltype_col,
            clone_col,
            match_col,
        ]
        missing_cols = [
            x for x in required_cols
            if x not in adata.obs.columns
        ]
        if missing_cols:
            raise KeyError(
                f"Missing columns in adata.obs: {missing_cols}"
            )

        # ----------------------------------------------------
        # SEP mapping
        # ----------------------------------------------------
        adata_match_id = (
            adata.obs[match_col]
            .astype("string")
            .str.strip()
        )
 
        adata.obs[sep_col] = adata_match_id.map(sep_map)

        n_spots = adata.n_obs
        n_sep = int(adata.obs[sep_col].notna().sum())

   
        annotation_df = pd.DataFrame({
            "Sample": sample_id,
            "cell_id": adata.obs_names.astype(str),
            "Clone": adata.obs[clone_col].astype(object).values,
            "SEP": adata.obs[sep_col].astype(object).values,
            "CellType": adata.obs[celltype_col].astype(object).values,
        })

        annotation_tables.append(annotation_df)
 
        img_low, W_low, H_low, scale = (
            make_lowres_rgba_he_background(
                adata,
                image_key="staining_image",
                mask_key="final_mask",
                bg_alpha=bg_alpha,
                bg_scale=bg_scale,
            )
        )

        adata = add_lowres_spatial(
            adata,
            scale=scale,
            spatial_key=spatial_key,
        )
 
        celltype_adata = filter_for_plot(adata, celltype_col,)
        plot_spatial_lowres_rgba(
            celltype_adata,
            celltype_col,
            celltype_colors,
            img_low,
            W_low,
            H_low,
            f"{sample_id}_CellType",
            outdir,
            spot_size=spot_size,
            dpi=dpi,
        )
 
        sep_adata = filter_for_plot(adata, sep_col,)
        plot_spatial_lowres_rgba(
            sep_adata,
            sep_col,
            SEP_colors,
            img_low,
            W_low,
            H_low,
            f"{sample_id}_SEP",
            outdir,
            spot_size=spot_size,
            dpi=dpi,
        )
        clone_adata = filter_for_plot(adata, clone_col,)

        plot_spatial_lowres_rgba(
            clone_adata,
            clone_col,
            clone_colors,
            img_low,
            W_low,
            H_low,
            f"{sample_id}_Clone",
            outdir,
            spot_size=spot_size,
            dpi=dpi,
        )

        run_summary.append({
            "Sample": sample_id,
            "status": "OK",
            "n_spots": n_spots,
            "n_SEP_mapped": n_sep,
            "SEP_match_rate": n_sep / n_spots if n_spots > 0 else np.nan,
            "error": "",
        })

        print(
            f"[OK] {sample_id}: "
            f"{n_spots:,} spots; "
            f"SEP {n_sep:,}/{n_spots:,} "
            f"({n_sep / n_spots:.1%})"
        )

    except Exception as e:

        run_summary.append({
            "Sample": sample_id,
            "status": "FAILED",
            "n_spots": np.nan,
            "n_SEP_mapped": np.nan,
            "SEP_match_rate": np.nan,
            "error": repr(e),
        })

        print(f"[FAILED] {sample_id}: {repr(e)}")

    finally:
        for obj_name in [
            "adata",
            "celltype_adata",
            "sep_adata",
            "clone_adata",
            "img_low",
        ]:
            if obj_name in globals():
                del globals()[obj_name]

        gc.collect()